# Phase-1 Data Completion — Colab (ephemeral `/content` only)

Completes the ICA-2026 Phase-1 **data prerequisites** entirely inside the Colab
temporary filesystem (`/content`). **No Google Drive is used.**

### How to use

1. **Files** (left panel) → **Upload** → upload **`diagnosis-to-decision.zip`**
   (it lands at `/content/diagnosis-to-decision.zip`).
2. **Runtime → Run all.**
3. When it finishes, **`/content/phase1_return_package.zip`** downloads
   automatically — unpack it into your local repository.

The notebook unpacks the archive, auto-detects the repository root, installs the
package, resumes PlantDoc, materializes PlantVillage `color` pixels, runs the
cross-dataset leakage comparison + gate, runs the tests and Phase-1 checks, and
packages **only** reports / manifests / indexes for return (never raw images).

> Public dataset — **no tokens, no credentials, no personal paths**. This notebook
> never commits or pushes, and never trains a model.


## 1 · Fixed paths (nothing to edit)

In [ ]:
ARCHIVE_PATH   = "/content/diagnosis-to-decision.zip"
PROJECT_DIR    = "/content/diagnosis-to-decision"
RETURN_PACKAGE = "/content/phase1_return_package.zip"
THRESHOLD      = 6   # reviewed perceptual-hash leakage threshold (fixed; do not change)
print("ARCHIVE_PATH  :", ARCHIVE_PATH)
print("PROJECT_DIR   :", PROJECT_DIR)
print("RETURN_PACKAGE:", RETURN_PACKAGE)
print("THRESHOLD     :", THRESHOLD)


## 2 · Check the archive and free space

In [ ]:
import os, shutil
if not os.path.exists(ARCHIVE_PATH):
    raise SystemExit("Upload diagnosis-to-decision.zip through the Colab Files panel, "
                     "then select Runtime → Run all.")
archive_bytes = os.path.getsize(ARCHIVE_PATH)
free_bytes    = shutil.disk_usage("/content").free
need_bytes    = archive_bytes * 3  # archive + extracted + working headroom
print(f"archive size   : {archive_bytes/1e6:.1f} MB")
print(f"free in /content: {free_bytes/1e9:.1f} GB")
print(f"approx. needed  : {need_bytes/1e6:.1f} MB")
if free_bytes < need_bytes:
    raise SystemExit(f"Not enough free space in /content: need ~{need_bytes/1e6:.0f} MB, "
                     f"have {free_bytes/1e6:.0f} MB.")


## 3 · Unpack the archive and auto-detect the repository root

- Skips extraction if the project is already unpacked and a valid root is found.
- **Never deletes** an existing `/content/diagnosis-to-decision`.
- Finds the real root even if the ZIP added an extra nested folder; if several
  `pyproject.toml` exist, picks the directory that also contains `PROJECT_BRIEF.md`
  and `scripts/run_phase1_data_completion.py`. Sets `REPO_DIR` and `DATA_DIR`.

In [ ]:
import glob, zipfile

def find_repo_dir(base):
    if not os.path.isdir(base):
        return None
    pyprojects = sorted(glob.glob(os.path.join(base, "**", "pyproject.toml"), recursive=True),
                        key=lambda p: p.count(os.sep))
    roots = [os.path.dirname(p) for p in pyprojects]
    def strong(d):
        return (os.path.exists(os.path.join(d, "PROJECT_BRIEF.md")) and
                os.path.exists(os.path.join(d, "scripts", "run_phase1_data_completion.py")))
    for d in roots:
        if strong(d):
            return d
    for d in roots:
        if os.path.exists(os.path.join(d, "PROJECT_BRIEF.md")):
            return d
    return roots[0] if roots else None

REPO_DIR = find_repo_dir(PROJECT_DIR)
if REPO_DIR:
    print("Project already unpacked; skipping extraction ->", REPO_DIR)
else:
    os.makedirs(PROJECT_DIR, exist_ok=True)  # never removes an existing PROJECT_DIR
    print("extracting", ARCHIVE_PATH, "->", PROJECT_DIR)
    with zipfile.ZipFile(ARCHIVE_PATH) as zf:
        zf.extractall(PROJECT_DIR)
    REPO_DIR = find_repo_dir(PROJECT_DIR)

if not REPO_DIR:
    raise SystemExit("Could not find a repository root (pyproject.toml + PROJECT_BRIEF.md + "
                     "scripts/run_phase1_data_completion.py) inside the archive.")

DATA_DIR = os.path.join(REPO_DIR, "data")
os.makedirs(DATA_DIR, exist_ok=True)
print("REPO_DIR:", REPO_DIR)
print("DATA_DIR:", DATA_DIR)


## 4 · Verify required files and install the package

In [ ]:
import subprocess, sys
required = ["PROJECT_BRIEF.md", "pyproject.toml",
            os.path.join("scripts", "run_phase1_data_completion.py"),
            os.path.join("scripts", "run_phase1_checks.sh")]
missing = [r for r in required if not os.path.exists(os.path.join(REPO_DIR, r))]
if missing:
    raise SystemExit("Missing required files: " + ", ".join(missing))
for r in required:
    print("  OK", r)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + "[hf]"], check=True)
print("installed ica26 from", REPO_DIR)
DRIVER = os.path.join(REPO_DIR, "scripts", "run_phase1_data_completion.py")


## 5 · Complete PlantDoc + materialize PlantVillage `color` pixels

The long step (PlantVillage `data.zip` ≈ 2 GB). Re-running **resumes** — present files are skipped, partial downloads continue. Uses the project's existing driver and fixed dataset revisions / threshold.

In [ ]:
r_data = subprocess.run([sys.executable, DRIVER,
                         "--repo-dir", REPO_DIR, "--data-dir", DATA_DIR,
                         "--steps", "plantdoc,plantvillage", "--threshold", str(THRESHOLD)])
print("data-completion step exit:", r_data.returncode)


## 6 · Cross-dataset leakage comparison + duplicate disposition + gate

In [ ]:
r_leak = subprocess.run([sys.executable, DRIVER,
                         "--repo-dir", REPO_DIR, "--data-dir", DATA_DIR,
                         "--steps", "leakage,gate", "--threshold", str(THRESHOLD), "--no-download"])
print("leakage+gate step exit:", r_leak.returncode)


## 7 · Tests + Phase-1 checks

`run_phase1_checks.sh` exit code **2 = PARTIAL/BLOCKED** (not a crash) — the code is recorded and shown in the summary.

In [ ]:
r_pytest = subprocess.run([sys.executable, "-m", "pytest", "-q"],
                          cwd=REPO_DIR, capture_output=True, text=True)
PYTEST_EXIT = r_pytest.returncode
print(r_pytest.stdout[-1500:])
print("pytest exit:", PYTEST_EXIT)

r_checks = subprocess.run(["bash", os.path.join(REPO_DIR, "scripts", "run_phase1_checks.sh")],
                          cwd=REPO_DIR, env=dict(os.environ, PYTHON=sys.executable),
                          capture_output=True, text=True)
CHECKS_EXIT = r_checks.returncode          # 0=PASS, 2=PARTIAL/BLOCKED, 1=FAIL (2 is NOT a crash)
CHECKS_OUT = r_checks.stdout
print(CHECKS_OUT[-1500:])
print("phase-1 checks exit:", CHECKS_EXIT, "(0=PASS, 2=PARTIAL/BLOCKED, 1=FAIL)")


## 8 · Final summary + return package (auto-download)

Packages **only** the report/manifest/index/exclusion artifacts listed below — **never** `data/raw/`, images, dataset archives, `.git`, caches or partial files.

In [ ]:
import json, re, zipfile

def load(rel):
    fp = os.path.join(REPO_DIR, rel)
    try:
        return json.load(open(fp)) if os.path.exists(fp) else None
    except Exception:
        return None

# ---- counts ----
def count_images(root):
    n = 0
    if os.path.isdir(root):
        for dp, _, fns in os.walk(root):
            n += sum(f.lower().endswith((".jpg", ".jpeg", ".png")) for f in fns)
    return n

pd_s   = load("data/manifests/plantdoc_summary.json") or {}
pv_s   = load("data/manifests/plantvillage_summary.json") or {}
gate   = load("reports/leakage_gate.json") or {}
leak   = load("reports/leakage_plantvillage_vs_plantdoc_summary.json") or {}
pd_rec = pd_s.get("reconciliation", {})
pv_ver = pv_s.get("pixel_verification", {})

pd_fs  = count_images(os.path.join(DATA_DIR, "raw", "plantdoc"))
pv_fs  = pv_ver.get("filesystem_image_count",
                    count_images(os.path.join(DATA_DIR, "raw", "plantvillage", "extracted")))

m = re.search(r"OVERALL:\s*(\w+)", CHECKS_OUT or "")
checks_verdict = m.group(1) if m else "UNKNOWN"
pm = re.search(r"(\d+)\s+passed", r_pytest.stdout or "")
pytest_passed = pm.group(1) if pm else "?"

exact   = leak.get("n_exact_pairs")
near    = leak.get("n_near_pairs")
pending = leak.get("n_pending_review")

blockers = []
if not pd_s.get("complete"):
    blockers.append(f"PlantDoc incomplete ({pd_rec.get('downloaded_image_count')}/{pd_rec.get('upstream_repository_count')})")
if not pv_s.get("pixels_materialized"):
    blockers.append("PlantVillage pixels not materialized")
if gate.get("status") != "pass":
    blockers.append(f"leakage gate status={gate.get('status')}")
if pending:
    blockers.append(f"{pending} near-duplicate pair(s) pending review")
blockers.append("mapping has 0 approved rows (human review required — by design)")

print("=" * 64)
print("PHASE-1 DATA COMPLETION — FINAL SUMMARY")
print("=" * 64)
print(f"PlantDoc     filesystem/valid/manifest : {pd_fs} / {pd_rec.get('valid_decodable_count')} / {pd_rec.get('manifest_count')}")
print(f"PlantVillage filesystem/valid/manifest : {pv_fs} / {pv_ver.get('valid_decodable_count')} / {pv_s.get('n_images')}")
print(f"PlantVillage pixels_materialized       : {pv_s.get('pixels_materialized')}")
print(f"pytest                                 : {pytest_passed} passed, exit {PYTEST_EXIT}")
print(f"Phase-1 checks                         : {checks_verdict}, exit {CHECKS_EXIT}")
print(f"cross-dataset exact / near / pending   : {exact} / {near} / {pending}")
print(f"leakage gate status                    : {gate.get('status')}")
print("remaining blockers:")
for b in blockers:
    print("   -", b)

# ---- return package (only listed artifacts; raw data / images / archives excluded) ----
report_files = [
    "reports/PHASE1_DATA_COMPLETION_REPORT.md",
    "reports/leakage_gate.json",
    "reports/leakage_plantvillage_vs_plantdoc_pairs.csv",
    "reports/leakage_plantvillage_vs_plantdoc_summary.json",
    "reports/CROSS_DATASET_NEAR_DUPLICATE_REVIEW.md",
    "reports/leakage_gate_guard_test.json",
    "reports/PLANTDOC_COUNT_RECONCILIATION.md",
]
data_dirs = ["data/manifests", "data/exclusions", "data/indexes", "data/interim"]
SKIP_SUFFIX = (".jpg", ".jpeg", ".png", ".zip", ".pyc")

def excluded(rel):
    parts = rel.replace("\\", "/").split("/")
    if rel.replace("\\", "/").startswith("data/raw/"):
        return True
    if any(seg in (".git", ".venv", "__pycache__") for seg in parts):
        return True
    if rel.lower().endswith(SKIP_SUFFIX):
        return True
    if os.path.basename(rel).endswith((".part", ".incomplete", ".tmp")):
        return True
    return False

added = 0
with zipfile.ZipFile(RETURN_PACKAGE, "w", zipfile.ZIP_DEFLATED) as zf:
    for rel in report_files:
        ap = os.path.join(REPO_DIR, rel)
        if os.path.exists(ap) and not excluded(rel):
            zf.write(ap, rel); added += 1
    for d in data_dirs:
        base = os.path.join(REPO_DIR, d)
        if not os.path.isdir(base):
            continue
        for dp, _, fns in os.walk(base):
            for f in fns:
                ap = os.path.join(dp, f)
                rel = os.path.relpath(ap, REPO_DIR)
                if not excluded(rel):
                    zf.write(ap, rel); added += 1

size_mb = os.path.getsize(RETURN_PACKAGE) / 1e6
print(f"\nreturn package: {RETURN_PACKAGE}  ({added} files, {size_mb:.2f} MB)")

from google.colab import files
files.download(RETURN_PACKAGE)
